In [ ]:
import requests
import polars as pl

def get_prices(hub:bool):
    # Request
    url = "https://api.misoenergy.org/MISORTWDDataBroker/DataBrokerServices.asmx"
    headers = {
        "Content-Type": "text/xml; charset=utf-8",
        "SOAPAction": "http://tempuri.org/MethodName" 
    }
    if hub:
        payload = r'{"messageType":"GetDataByNodeTypes","clientMessage":{"nodeTypes":["HUB"]}}'
    else:
        payload = r'{"messageType":"GetDataByNodeTypes","clientMessage":{"nodeTypes":["GEN","INT","LZN"]}}'

    # Response
    try:
        response = requests.post(url, headers=headers, data=payload, timeout=10)
        
        response.raise_for_status()
        print(f"Status Code: {response.status_code}")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        # TODO: return empty df

    df = pl.DataFrame(response.json()["data"])
    if "NSI" in df.columns:
        df = df.drop("NSI")
    df = df.rename({
        col: col.lower()
        for col in df.columns
    })
    df = df.select(["location", "lmp", "mcc", "mlc"])
    return df

In [18]:
nodes = get_prices(hub=False)
hubs = get_prices(hub=True)
prices = pl.concat([nodes, hubs]).sort(by="location")

Status Code: 200
Status Code: 200


In [19]:
prices

location,lmp,mcc,mlc
str,f64,f64,f64
"""AECI""",19.61,-16.62,-2.08
"""ALTE.1ROCKGEN""",41.75,5.05,-1.61
"""ALTE.COLUMBAL1""",44.96,7.0,-0.35
"""ALTE.COLUMBAL2""",44.81,7.0,-0.5
"""ALTE.EDGG5G5""",44.8,6.53,-0.04
…,…,…,…
"""WPS.COLUMBIA2""",44.81,7.0,-0.5
"""WPS.DPC.WESTN4""",51.35,9.4,3.64
"""WPS.WESTON3""",51.13,9.41,3.41
